In [6]:
import pandas as pd

In [8]:
df = pd.read_csv('dirty_financial_transactions.csv')

df['Transaction_Date'] = pd.to_datetime(df['Transaction_Date'], format='mixed', errors='coerce')
df = df.sort_values('Transaction_Date').reset_index(drop=True)

split = int(len(df) * 0.9) #Split the original dataset into two distinct files/tables
df_initial = df.iloc[:split].copy() #Initial Load: The bulk of the historical data
df_delta = df.iloc[split:].copy() #Secondary Load (Delta): A smaller subset representing a new day/batch of data

duplicates = df_initial.sample(n=5, random_state=42).copy() #Exact duplicates of some rows from the initial load to simulate pipeline glitches
df_delta = pd.concat([df_delta, duplicates], ignore_index=True)

new_records = pd.DataFrame({ #New records that did not exist in the initial load
    'Transaction_ID': ['T_NEW_9991', 'T_NEW_9992'],
    'Transaction_Date': [pd.Timestamp('2026-08-26'), pd.Timestamp('2026-08-26')],
    'Customer_ID': ['C9999', 'C8888'],
    'Product_Name': ['Smartphone', 'Laptop'],
    'Quantity': [1.0, 2.0],
    'Price': [500.0, 1200.0],
    'Payment_Method': ['Cash', 'Paypal'],
    'Transaction_Status': ['completed', 'completed']
})

df_delta = pd.concat([df_delta, new_records], ignore_index=True)


df_delta.iloc[0, df_delta.columns.get_loc('Product_Name')] = 'Phone' #SCD Type 1 changes: Modify a non-historical attribute 

target_customer = df_initial['Customer_ID'].dropna().iloc[0]
df_delta.loc[df_delta['Customer_ID'] == target_customer, 'Payment_Method'] = 'PayPal_Updated' #SCD Type 2 changes: Modify a historical attribute 


df_initial.to_csv('initial_load.csv', index=False)
df_delta.to_csv('delta_load.csv', index=False)

1. Purpose of the three-layer DWH architecture:

If we connect Power BI straight to raw files, any small change in the file structure will instantly break all dashboards. Also, raw data is very dirty. The DWH layers clean this up and make sure everyone in the company looks at the same verified numbers (a Single Source of Truth). Plus, raw tables are not built for fast analytics, while Mart tables are ready for business use.


2. Why do reporting tools like Power BI perform significantly better with a Star Schema rather than a highly normalized relational model?

In a normalized 3NF model, data is split into many small tables. To get a simple report, the database has to do a lot of heavy JOIN operations between tables.
A Star Schema combines things into wider, flat dimension tables around one central fact table. Power BI's internal engine works much faster with this structure, compresses data better, and makes it much easier to write measures and formulas without needing to handle complex database links.

In [9]:
df_initial.info()

df_delta.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 90000 entries, 0 to 89999
Data columns (total 8 columns):
 #   Column              Non-Null Count  Dtype         
---  ------              --------------  -----         
 0   Transaction_ID      85462 non-null  object        
 1   Transaction_Date    31739 non-null  datetime64[ns]
 2   Customer_ID         85626 non-null  object        
 3   Product_Name        90000 non-null  object        
 4   Quantity            85480 non-null  float64       
 5   Price               59896 non-null  object        
 6   Payment_Method      90000 non-null  object        
 7   Transaction_Status  75038 non-null  object        
dtypes: datetime64[ns](1), float64(1), object(6)
memory usage: 5.5+ MB
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10007 entries, 0 to 10006
Data columns (total 8 columns):
 #   Column              Non-Null Count  Dtype         
---  ------              --------------  -----         
 0   Transaction_ID      9526 non-null   